Restarted hetker (Python 3.13.3)

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is on sys.path for local imports
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
import cmdstanpy as csp
import seaborn as sns
from scipy.special import logit, expit
import matplotlib.pyplot as plt
import networkx as nx


from src.data_processing.ORM import ContagionMetadata
from src.data_processing.database_utils import decompress_array, get_timeseries, get_adjacency_matrix

def cap_exposure_bins(M, N, max_k):
    """
    Collapse exposure matrices to `max_k` bins by pooling any counts above the cap
    into the final bin (>= max_k).
    """
    if max_k is None or max_k <= 0:
        return M, N

    def _cap(arr):
        if arr is None:
            return None
        arr = np.asarray(arr)
        if arr.shape[1] <= max_k:
            return arr[:, :max_k]
        capped = np.zeros((arr.shape[0], max_k), dtype=arr.dtype)
        if max_k > 1:
            capped[:, :max_k - 1] = arr[:, :max_k - 1]
        capped[:, -1] = arr[:, max_k - 1:].sum(axis=1)
        return capped

    return _cap(M), _cap(N)

# Database connection
db_path = project_root / "data" / "contagion.db"
engine = create_engine(f"sqlite:///{db_path}")
Session = sessionmaker(bind=engine)
session = Session()

/Users/willthompson/Documents/CSDS/side_projects/heterogeneous-infection-kernels/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
target_run_id = "oo_sim_165_duration"
run_metadata = session.query(ContagionMetadata).filter_by(run_id=target_run_id).first()

M = decompress_array(run_metadata.exposure_success_blob).astype(int)
N = decompress_array(run_metadata.exposure_failure_blob).astype(int)

# Cap high-k bins to reduce sparsity
kmax_cap = 20
M, N = cap_exposure_bins(M, N, max_k=kmax_cap)

print(f"Loaded Run: {run_metadata.run_id}")
num_nodes, kmax = M.shape
data = {
    "V": num_nodes,
    "kmax": kmax,
    "K": 2,     # Number of modes
    "M": M,
    "N": N,
    "DEVIATION_SCALE": [1.0, 1.0], # Hyperparameter
    "REPLICATES": 1000         # Number of replicates
    # "AVERAGE_SCALE": 2.0   # Unused in mixture model
}

OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/20/e3q8)